# 23 · Online Learning, Semi-Supervised y Active Learning

No todos los problemas tienen un dataset fijo y perfectamente etiquetado. A veces los datos llegan en stream, las etiquetas son caras o solo una fracción está etiquetada.

## Objetivos
- Entender `partial_fit` y aprendizaje incremental.
- Diferenciar batch, online y continual learning.
- Introducir semi-supervised learning con pseudo-labeling/label propagation.
- Diseñar active learning por incertidumbre.
- Relacionar estos enfoques con drift y costo de etiquetado.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import make_classification, make_moons
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier
from sklearn.svm import SVC
SEED=42; rng=np.random.default_rng(SEED)

## 1. Online / incremental learning
Batch retraining carga todo el dataset. Online learning actualiza el modelo en pequeños lotes. Es útil con streams, datasets enormes o cambios frecuentes.

`SGDClassifier.partial_fit` actualiza parámetros sin olvidar completamente el estado previo. Otros ecosistemas: River, Vowpal Wabbit, creme/River, online boosting.


In [ ]:
X,y=make_classification(n_samples=10000,n_features=25,n_informative=10,random_state=SEED)
idx=rng.permutation(len(X)); X,y=X[idx],y[idx]
scaler=StandardScaler(); clf=SGDClassifier(loss='log_loss',random_state=SEED)
acc=[]
for start in range(0,8000,500):
 xb,yb=X[start:start+500],y[start:start+500]
 scaler.partial_fit(xb); clf.partial_fit(scaler.transform(xb),yb,classes=np.array([0,1]))
 acc.append(clf.score(scaler.transform(X[8000:]),y[8000:]))
plt.plot(np.arange(1,len(acc)+1)*500,acc,'o-'); plt.xlabel('ejemplos procesados'); plt.ylabel('test accuracy'); plt.show()

## 2. Drift y forgetting
Si el mundo cambia, acumular todo el pasado puede perjudicar. Estrategias: ventanas móviles, decaimiento de pesos, detectores ADWIN/DDM, ensembles adaptativos y retraining condicionado. En neural networks aparece **catastrophic forgetting** en continual learning; técnicas como replay/EWC ayudan.


## 3. Semi-Supervised Learning
Tenemos pocos labels y muchos ejemplos sin etiqueta. Supuestos típicos: puntos cercanos/cluster comparten label; la estructura de $P(X)$ informa sobre $P(Y|X)$. Si esos supuestos fallan, pseudo-labels pueden amplificar errores.


In [ ]:
X,y=make_moons(n_samples=1200,noise=.18,random_state=SEED); labels=y.copy(); unlabeled=rng.choice(len(y),size=1000,replace=False); labels[unlabeled]=-1
ls=LabelSpreading(kernel='rbf',gamma=20,alpha=.2,max_iter=60).fit(X,labels); print('LabelSpreading accuracy real',accuracy_score(y,ls.transduction_))
plt.scatter(X[:,0],X[:,1],c=ls.transduction_,s=12); plt.title('Etiquetas propagadas'); plt.show()

## 4. Self-training / pseudo-labeling
Entrenas con labels conocidos, predices unlabeled y agregas solo predicciones de alta confianza. Iteras. La calibración importa: un modelo overconfident puede agregar etiquetas incorrectas.


In [ ]:
base=SVC(probability=True,gamma='scale'); st=SelfTrainingClassifier(base,threshold=.9,max_iter=15).fit(X,labels); print('self-training accuracy',accuracy_score(y,st.predict(X)),'iteraciones',st.n_iter_)

## 5. Active Learning
En vez de etiquetar aleatoriamente, el modelo solicita labels de ejemplos informativos. Estrategia básica: uncertainty sampling (probabilidad cercana a 0.5). Otras: margin sampling, query-by-committee, expected model change, diversity-aware sampling.


In [ ]:
# simulación de active learning
X,y=make_classification(n_samples=2500,n_features=12,n_informative=6,random_state=SEED); pool=np.arange(len(y)); labeled=list(rng.choice(pool,40,replace=False)); remaining=np.setdiff1d(pool,labeled)
history=[]
for round_ in range(12):
 m=SGDClassifier(loss='log_loss',random_state=SEED).fit(X[labeled],y[labeled]); prob=m.predict_proba(X[remaining])[:,1]; uncertainty=np.abs(prob-.5); query_idx=np.argsort(uncertainty)[:30]; new=remaining[query_idx]; labeled.extend(new.tolist()); remaining=np.setdiff1d(remaining,new); history.append(m.score(X,y))
plt.plot(np.arange(len(history))*30+40,history,'o-'); plt.xlabel('labels usados'); plt.ylabel('accuracy sobre dataset (demo)'); plt.show()

## 6. Weak supervision
Otra opción es crear labels aproximados con reglas, heurísticas, modelos previos o fuentes externas y combinarlos probabilísticamente. Snorkel popularizó labeling functions. Útil cuando expertos pueden escribir reglas más rápido que etiquetar miles de filas.

## 7. Self-supervised learning
En deep learning moderno, el modelo crea una tarea de preentrenamiento a partir de datos sin labels: predecir tokens enmascarados, siguiente token, contrastive learning, masked autoencoders. Esto alimenta BERT, GPT, CLIP y gran parte de foundation models; lo veremos en IA.

## Errores comunes
- pseudo-labeling con confianza mal calibrada;
- active learning que consulta solo casos raros y pierde diversidad;
- online learning sin monitorear drift;
- usar unlabeled de una distribución distinta;
- olvidar que labels humanos también tienen ruido.

## Ejercicios
1. Compara active learning vs random sampling con igual presupuesto.
2. Usa River para un stream con concept drift.
3. Simula drift abrupto y gradual.
4. Implementa pseudo-labeling manual.
5. Agrega diversity sampling con K-Means.
6. Investiga FixMatch, Mean Teacher y contrastive self-supervised learning.
